# MOP 3231 · Week 2 Lab / Task 1 — Data Preprocessing & Cleaning

**Student:** Imamsidikov
**Dataset:** `data/almaty_apartments_raw.csv` (synthetic Almaty apartment listings, generated for the course — not market data)

The notebook is organised in 8 steps:

| # | Step |
|---|------|
| 1 | Project setup & versions |
| 2 | Load and inspect |
| 3 | Clean and transform (duplicates/reposts, districts, text → numbers) |
| 4 | Sentinels, impossible values, outliers → `data/almaty_apartments_clean.csv` |
| 5 | Missing data (split first, diagnose, masking experiment, fill) |
| 6 | Scaling and encoding (+ sanity-check model) |
| 7 | EDA and cleaning log |
| 8 | Summary and git history |

## Step 1 · Project setup

Isolated venv `.Imamsidikov_ML_task1` (Python 3.12), pinned `requirements.txt`, git repository.
Jupyter is only a tool and is not listed in `requirements.txt`.

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib

print("Python      ", sys.version.split()[0])
print("NumPy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("matplotlib  ", matplotlib.__version__)

Python       3.12.5
NumPy        2.0.2
pandas       2.3.3
scikit-learn 1.6.1
matplotlib   3.9.4


In [2]:
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# the cleaning log: one entry per operation (step, operation, rows before, rows after, cells changed)
LOG = []

def log(step, operation, before, after, cells=0):
    LOG.append({"Step": step, "Operation": operation, "Rows before": before,
                "Rows after": after, "Cells changed": int(cells)})
    print(f"[{step}] {operation}: {before} -> {after} rows, {int(cells)} cells changed")

## Step 2 · Load and inspect

**YOUR_ID = 1234** — ⚠️ placeholder: replace with the last four digits of my student ID, used as `random_state` for the 90 % sample and for the train/test split.

In [3]:
YOUR_ID = 1234

raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=YOUR_ID)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print("raw file:", raw.shape, "-> my sample:", df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

raw file: (1284, 11) -> my sample: (1153, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           53
building_type         0
ceiling_height_m    130
price_kzt             0
dtype: int64
                   count unique         top freq         mean         std  min     25%     50%     75%     max
listing_id          1153   1080   ALM-10932    3          NaN         NaN  NaN     NaN     NaN     NaN     NaN
listed_date         1153    246  2026-04-18   11          NaN         NaN  NaN     NaN     NaN     NaN     NaN
district            11

### Q1 · Types — raw values of the object columns

In [4]:
for col in ["rooms", "area_m2", "price_kzt", "listed_date"]:
    print(col, df[col].sample(10, random_state=YOUR_ID).tolist())

def non_numeric(s):
    s = s.astype(str)
    return s[~s.str.fullmatch(r"-?\d+(\.\d+)?")]

for col in ["rooms", "area_m2", "price_kzt"]:
    bad = non_numeric(df[col])
    print(f"\n{col}: {len(bad)} non-numeric values, e.g.", bad.drop_duplicates().head(6).tolist())

rooms ['1', '2', '2', '1', '3', 'studio', '3', '2', '4', '3']
area_m2 ['39.5', '56.4', '60.2', '34.3', '94.4', '32.4', '72.4', '544.0', '103.2', '77.0']
price_kzt ['43250000', '26490000', '67870000', '20080000', '68640000', '23030000', '50990000', '25030000', '46030000', '94,030,000']
listed_date ['2026-06-29', '2026-03-29', '2026-05-11', '2026-03-20', '2026-08-08', '2026-07-11', '2026-02-05', '2026-08-24', '2026-02-01', '2026-01-14']

rooms: 30 non-numeric values, e.g. ['studio']

area_m2: 95 non-numeric values, e.g. ['59,4', '54.6 m²', '40,8', '62,2', '61.1 m²', '40.8 m²']

price_kzt: 89 non-numeric values, e.g. ['41,560,000', '29 060 000 ₸', '52,760,000', '31,620,000', '70,860,000', '32 620 000 ₸']


**Object columns that should be numbers or dates**

| Column | Why pandas did not parse it |
|---|---|
| `rooms` | contains the word `"studio"` next to the digits, so the whole column is text |
| `area_m2` | some values use a decimal **comma** (`"62,2"`) and some carry a unit suffix (`"61.1 m²"`) |
| `price_kzt` | thousands separators — commas (`"41,560,000"`) or spaces plus the currency sign (`"82 080 000 ₸"`) |
| `listed_date` | CSV has no date type; `read_csv` leaves `YYYY-MM-DD` strings as object unless told to parse them |

(`district`, `building_type`, `listing_id` are object on purpose — they are text/categorical.)
`year_built` and `ceiling_height_m` are `float64` rather than int only because they contain NaN.

### Q2 · Grain — exact duplicates and reposts

In [5]:
n_exact = df.duplicated().sum()
n_repost = df.drop_duplicates()["listing_id"].duplicated().sum()
print("(a) exact copies of another row:              ", n_exact)
print("(b) not exact copies but share a listing_id:  ", n_repost)

dedup = df.drop_duplicates()
reposts = dedup[dedup["listing_id"].duplicated(keep=False)].sort_values(["listing_id", "listed_date"])
two_ids = reposts["listing_id"].unique()[:2]
print(reposts[reposts["listing_id"].isin(two_ids)])

differs = {c: int((reposts.groupby("listing_id")[c].nunique() > 1).sum()) for c in df.columns if c != "listing_id"}
print("\nnumber of repost groups in which each column differs:", differs)

(a) exact copies of another row:               31
(b) not exact copies but share a listing_id:   42
    listing_id listed_date   district rooms area_m2  floor  total_floors  year_built building_type  ceiling_height_m price_kzt
971  ALM-10003  2026-05-04     Auezov     2    48.6      2             5      1978.0         panel              2.55  28320000
7    ALM-10003  2026-05-29     Auezov     2    48.6      2             5      1978.0         panel              2.55  26120000
945  ALM-10009  2025-12-03  Medeuskiy     1    33.4      2             9      1992.0         brick              3.19  42700000
178  ALM-10009  2026-01-19  Medeuskiy     1    33.4      2             9      1992.0         brick              3.19  39400000

number of repost groups in which each column differs: {'listed_date': 42, 'district': 0, 'rooms': 0, 'area_m2': 0, 'floor': 0, 'total_floors': 0, 'year_built': 0, 'building_type': 0, 'ceiling_height_m': 0, 'price_kzt': 42}


Within every repost pair **only `listed_date` and `price_kzt` differ** — the flat itself (district, rooms, area, floors, year, type, ceiling) is identical.
It is the same apartment posted again at a new asking price.

**Grain of the cleaned table:** *one row = one apartment listing (one unique `listing_id`) with its current asking price.*

### Q3 · Validation spec

| Column | Target type | Valid values |
|---|---|---|
| `listing_id` | string | `ALM-#####`, unique after cleaning |
| `listed_date` | datetime | 2025-11-01 … 2026-09-30 (the data window), not in the future |
| `district` | category | exactly one of the 8: Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu |
| `rooms` | int | 1 – 6 (studio = 1) |
| `area_m2` | float | 20 – 250 m², and plausible for the room count (≈ 25–60 m² per room) |
| `floor` | int | 1 – `total_floors` (**two-column rule: `floor ≤ total_floors`**) |
| `total_floors` | int | 1 – 40 |
| `year_built` | int | 1930 – 2026 |
| `building_type` | category | panel / brick / monolith |
| `ceiling_height_m` | float | 2.4 – 4.0 m |
| `price_kzt` | int | > 0; 5 M – 500 M ₸; price per m² consistent with the district |